# Day 15. Transformer Block

**오늘의 목표**
- 지금까지 만든 부품들이 GPT 안에서 **어디에 들어가는지** 큰 그림을 그린다
- Transformer Block = **소통(Attention) + 생각(FFN)** 이라는 구조를 설명할 수 있다
- 원본 Transformer의 **Post-LN**과 GPT-2 이후의 **Pre-LN**이 어떻게 다른지 안다
- `Block(nn.Module)`을 직접 만들고, 입력과 출력 shape가 같아서 **여러 개를 쌓을 수 있다**는 걸 확인한다

> 오늘은 큰 그림을 그리는 날입니다. Residual Connection과 LayerNorm은 **Day 16**, FFN은 **Day 17**에서 자세히 파고듭니다.
> 오늘은 "이런 역할을 하는 부품이구나" 정도로만 이해해도 충분해요.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False
torch.manual_seed(0)

def r(t, d=3):
    # 텐서를 보기 좋게 반올림한 리스트로
    return [round(v, d) + 0.0 for v in t.tolist()]   # +0.0: -0.0 표시 방지

def count(module):
    # 모듈의 파라미터 개수
    return sum(p.numel() for p in module.parameters())

## 1. 지금 어디까지 왔나

2주 동안 만든 것들을 GPT 전체 흐름에 놓아보면 이렇습니다.

```text
"나는 오늘 학교에 간다"
        ↓  Tokenizer                    (Day 8)
Token ID       (B, T)
        ↓  Token Embedding              (Day 9)
        +  Position Embedding           (Day 10)
x              (B, T, C)
        ↓
   ┌─────────────────────┐
   │        ???          │   ← 오늘 채울 부분: Transformer Block × N
   └─────────────────────┘
        ↓
x              (B, T, C)
        ↓  lm_head (Linear C → V)       (Day 3)
logits         (B, T, V)
        ↓  Softmax                      (Day 5)
다음 Token 확률
```

Day 14에서 만든 `CausalSelfAttention`이 `???` 안에 들어가는 핵심 부품이지만, 그것만으로는 부족합니다.
오늘은 이 상자 하나, **Transformer Block**을 완성합니다.

## 2. Block = 소통 + 생각

Transformer Block은 크게 **두 단계**로 이루어져 있습니다.

| 단계 | 부품 | 하는 일 | 정보가 흐르는 방향 |
|---|---|---|---|
| ① 소통 (communication) | **Self-Attention** | Token들이 서로 정보를 주고받음 | **T 축을 가로질러** (Token ↔ Token) |
| ② 생각 (computation) | **FFN** (Feed Forward) | 각 Token이 받은 정보를 **혼자서** 가공 | **C 축 안에서만** (Token 하나씩 따로) |

비유하면:
```text
① 회의 시간 (Attention)  : 다 같이 모여서 서로의 이야기를 듣는다
② 개인 작업 (FFN)        : 자리로 돌아가서 들은 내용을 각자 정리한다
```

이걸 여러 번 반복하는 게 Transformer입니다. (회의 → 정리 → 회의 → 정리 → ...)

여기에 학습을 안정시키는 부품 두 개가 붙습니다.
- **Residual Connection**: 입력을 그대로 더해서 원래 정보를 보존 (`x + f(x)`) → Day 16
- **LayerNorm**: 각 Token 벡터의 크기를 일정하게 맞춤 → Day 16

### 2-1. FFN은 정말 Token끼리 섞지 않을까? 직접 확인

In [ ]:
C = 8
ffn = nn.Sequential(nn.Linear(C, 4 * C), nn.GELU(), nn.Linear(4 * C, C))

x = torch.randn(1, 4, C)            # Token 4개
x_changed = x.clone()
x_changed[0, 3] += 10.0             # 마지막 Token만 크게 바꿈

out1, out2 = ffn(x), ffn(x_changed)
for t in range(4):
    same = torch.allclose(out1[0, t], out2[0, t])
    print(f"Token {t} 출력 {'그대로' if same else '바뀜'}")

마지막 Token을 바꿔도 **다른 Token의 FFN 출력은 그대로**입니다. FFN은 Token끼리 정보를 섞지 않아요.
그래서 Token끼리 정보를 주고받는 건 **오직 Attention의 역할**입니다. 둘 다 필요한 이유예요.

## 3. Post-LN vs Pre-LN

### 3-1. 커리큘럼 4-1의 구조 = 원본 Transformer (2017, Post-LN)

커리큘럼 4-1에 나온 그림은 원본 논문 *Attention Is All You Need* 방식입니다.
**LayerNorm이 더하기(Residual) 뒤에** 옵니다.

```text
x
├──────────────┐
↓              │
Attention      │
↓              │
+ ←────────────┘
↓
LayerNorm          ← 더한 다음에 정규화
├──────────────┐
↓              │
FFN            │
↓              │
+ ←────────────┘
↓
LayerNorm
```

코드로 쓰면:
```python
x = ln_1(x + attn(x))
x = ln_2(x + ffn(x))
```

### 3-2. GPT-2 이후 = Pre-LN ✅ 우리가 쓸 방식

GPT-2부터는 **LayerNorm을 각 부품 앞으로** 옮겼습니다.

```text
x
├──────────────┐
↓              │
LayerNorm      │   ← 먼저 정규화하고
↓              │
Attention      │
↓              │
+ ←────────────┘   ← 원래 x에 더함 (x는 정규화 안 된 그대로)
├──────────────┐
↓              │
LayerNorm      │
↓              │
FFN            │
↓              │
+ ←────────────┘
```

코드로 쓰면 (nanoGPT와 똑같은 형태):
```python
x = x + attn(ln_1(x))
x = x + ffn(ln_2(x))
```

### 3-3. 무엇이 다른가?

| | Post-LN (2017 원본) | Pre-LN (GPT-2 이후) |
|---|---|---|
| LayerNorm 위치 | 더한 **뒤** | 부품 **앞** |
| Residual 길 | 중간중간 LayerNorm을 통과 | 처음부터 끝까지 **아무것도 안 거치는 직통 길** |
| 층을 깊게 쌓으면 | 학습이 불안정해지기 쉬움 (학습률 warmup 필수) | 훨씬 안정적 |
| 마지막에 | 이미 정규화됨 | **마지막 LayerNorm(`ln_f`)을 한 번 더** 붙임 |

핵심은 **Residual 직통 길**입니다. Pre-LN에서는 `x`가 `+`만 거치면서 맨 아래부터 맨 위까지 그대로 흘러갑니다.
이 길 덕분에 층을 수십 개 쌓아도 Gradient가 잘 전달돼요. (자세한 이유는 Day 16)

> 💡 커리큘럼 6-8의 구조 그림(`Attention → Residual → FFN → Residual`)에는 LayerNorm이 빠져 있습니다.
> 실제로 구현할 때는 오늘 만드는 것처럼 **LayerNorm 두 개를 꼭 넣어야** 해요.

## 4. 부품 준비

### 4-1. CausalSelfAttention (Day 14에서 만든 것)

Day 14에서 완성한 클래스를 그대로 가져왔습니다.

In [ ]:
class CausalSelfAttention(nn.Module):

    def __init__(self, n_embd, n_head, block_size):
        super().__init__()
        assert n_embd % n_head == 0, "n_embd는 n_head로 나누어떨어져야 합니다"
        self.n_head = n_head
        self.c_attn = nn.Linear(n_embd, 3 * n_embd, bias=False)   # Q, K, V 한 번에
        self.proj   = nn.Linear(n_embd, n_embd)                   # Head 합친 뒤 정리
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape
        hs = C // self.n_head

        q, k, v = self.c_attn(x).split(C, dim=2)
        q = q.view(B, T, self.n_head, hs).transpose(1, 2)        # (B, nh, T, hs)
        k = k.view(B, T, self.n_head, hs).transpose(1, 2)
        v = v.view(B, T, self.n_head, hs).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) * hs ** -0.5             # (B, nh, T, T)
        att = att.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)

        y = att @ v                                              # (B, nh, T, hs)
        y = y.transpose(1, 2).contiguous().view(B, T, C)         # (B, T, C)
        return self.proj(y)

### 4-2. MLP (= FFN)

Day 4에서 본 `Linear(×4) → GELU → Linear(÷4)` 구조입니다. GPT-2 코드에서는 FFN을 **MLP**라고 부릅니다. (같은 것)

In [ ]:
class MLP(nn.Module):

    def __init__(self, n_embd):
        super().__init__()
        self.fc   = nn.Linear(n_embd, 4 * n_embd)
        self.gelu = nn.GELU()
        self.proj = nn.Linear(4 * n_embd, n_embd)

    def forward(self, x):
        return self.proj(self.gelu(self.fc(x)))

### 4-3. LayerNorm 맛보기

`nn.LayerNorm(C)`는 **각 Token 벡터(C개 숫자)마다 따로** 평균 0, 분산 1로 맞춰줍니다. shape는 바뀌지 않아요.
원리는 Day 16에서 직접 구현해봅니다.

In [ ]:
ln = nn.LayerNorm(8)
x = torch.randn(1, 3, 8) * 5 + 3          # 일부러 크고 치우친 값

y = ln(x)
print("LayerNorm 전 - Token별 평균:", r(x.mean(dim=-1)[0], 2), " 표준편차:", r(x.std(dim=-1, unbiased=False)[0], 2))
print("LayerNorm 후 - Token별 평균:", r(y.mean(dim=-1)[0], 2), " 표준편차:", r(y.std(dim=-1, unbiased=False)[0], 2))
print("shape:", tuple(x.shape), "→", tuple(y.shape))

## 5. Block 만들기 (Pre-LN)

부품 네 개를 조립합니다.

In [ ]:
class Block(nn.Module):

    def __init__(self, n_embd, n_head, block_size):
        super().__init__()
        self.ln_1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, block_size)
        self.ln_2 = nn.LayerNorm(n_embd)
        self.mlp  = MLP(n_embd)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))    # ① 소통 (Token끼리)
        x = x + self.mlp(self.ln_2(x))     # ② 생각 (Token 혼자)
        return x

`forward`가 단 **두 줄**입니다. 2주 동안 배운 Attention이 한 줄에 들어가 있어요.

### 5-1. shape 따라가기

`forward` 안에서 일어나는 일을 한 단계씩 풀어서 shape를 찍어봅시다. (학습 원칙 3: 직접 값을 출력한다)

In [ ]:
B, T, n_embd, n_head, block_size = 2, 5, 32, 4, 16
block = Block(n_embd, n_head, block_size)

x = torch.randn(B, T, n_embd)
print(f"입력 x               : {tuple(x.shape)}")

# ① 소통
h = block.ln_1(x);         print(f"  ln_1(x)            : {tuple(h.shape)}")
h = block.attn(h);         print(f"  attn(...)          : {tuple(h.shape)}")
x1 = x + h;                print(f"x + attn(ln_1(x))    : {tuple(x1.shape)}   ← Residual: 원래 x에 더함")

# ② 생각
h = block.ln_2(x1);        print(f"  ln_2(x)            : {tuple(h.shape)}")
h = block.mlp.fc(h);       print(f"  mlp.fc   (C → 4C)  : {tuple(h.shape)}   ← 잠깐 4배로 넓어짐")
h = block.mlp.gelu(h);     print(f"  mlp.gelu           : {tuple(h.shape)}")
h = block.mlp.proj(h);     print(f"  mlp.proj (4C → C)  : {tuple(h.shape)}")
x2 = x1 + h;               print(f"x + mlp(ln_2(x))     : {tuple(x2.shape)}   ← Residual")

print()
print("block(x)와 같은가?", torch.allclose(block(x), x2))

## 6. 가장 중요한 성질: 입력 shape == 출력 shape

Block에 `(B, T, C)`가 들어가면 `(B, T, C)`가 나옵니다.
그래서 **Block의 출력을 다음 Block의 입력으로 그대로** 넣을 수 있어요. (커리큘럼 4-5)

```text
Embedding → Block 1 → Block 2 → Block 3 → ... → Block N → lm_head
```

In [ ]:
n_layer = 4
blocks = nn.Sequential(*[Block(n_embd, n_head, block_size) for _ in range(n_layer)])

x = torch.randn(B, T, n_embd)
h = x
for i, blk in enumerate(blocks):
    h = blk(h)
    print(f"Block {i + 1} 통과 후: {tuple(h.shape)}")

print()
print("nn.Sequential로 한 번에:", tuple(blocks(x).shape))

GPT-2 small은 이 Block을 **12개**, GPT-3는 **96개** 쌓았습니다.
구조는 똑같고 **개수(`n_layer`)와 크기(`n_embd`)만** 다릅니다.

## 7. Block 하나의 파라미터는 어디에 몰려 있나

`C = n_embd`라고 하면:

| 부품 | 들어있는 것 | 파라미터 수 | 대략 |
|---|---|---|---|
| `ln_1`, `ln_2` | 각각 weight C + bias C | `2C × 2 = 4C` | 아주 작음 |
| `attn.c_attn` | Linear(C, 3C), bias 없음 | `3C²` | |
| `attn.proj` | Linear(C, C) | `C² + C` | |
| → Attention 합계 | | `4C² + C` | **≈ 4C²** |
| `mlp.fc` | Linear(C, 4C) | `4C² + 4C` | |
| `mlp.proj` | Linear(4C, C) | `4C² + C` | |
| → MLP 합계 | | `8C² + 5C` | **≈ 8C²** |

**MLP가 Attention의 약 2배**입니다. Block 파라미터의 약 2/3가 FFN에 있어요.

In [ ]:
block = Block(n_embd=32, n_head=4, block_size=16)

parts = {
    "LayerNorm ×2": count(block.ln_1) + count(block.ln_2),
    "Attention":    count(block.attn),
    "MLP (FFN)":    count(block.mlp),
}
total = count(block)
for name, n in parts.items():
    print(f"{name:13s}: {n:6d}개  ({n / total * 100:4.1f}%)")
print(f"{'합계':13s}: {total:6d}개")

plt.figure(figsize=(6, 3))
plt.barh(list(parts.keys()), list(parts.values()), color=["#BBBBBB", "#4C72B0", "#DD8452"])
for i, n in enumerate(parts.values()):
    plt.text(n, i, f" {n:,}", va="center")
plt.title("Block 하나의 파라미터 (n_embd = 32)")
plt.xlim(0, max(parts.values()) * 1.2)
plt.tight_layout()
plt.show()

GPT-2 small(`n_embd = 768`, Block 12개)로 계산하면:

In [ ]:
gpt2_block = Block(n_embd=768, n_head=12, block_size=1024)
per_block = count(gpt2_block)
print(f"Block 1개 : {per_block:,}개")
print(f"Block 12개: {per_block * 12:,}개   ← GPT-2 small 전체 1.24억 개 중 대부분")

## 8. Block을 통과해도 미래를 보지 않을까?

Day 14에서 Attention이 미래를 못 보게 Causal Mask를 넣었습니다.
그런데 Block에는 LayerNorm, MLP, Residual이 추가됐죠. 이것들이 **Token끼리 섞지 않으니까** 여전히 안전해야 합니다. 확인해봅시다.

In [ ]:
torch.manual_seed(0)
blocks = nn.Sequential(*[Block(32, 4, 16) for _ in range(3)])

x = torch.randn(1, 5, 32)
x_future = x.clone()
x_future[0, 4] = torch.randn(32)          # 마지막(미래) Token만 바꿈

out1, out2 = blocks(x), blocks(x_future)
for t in range(5):
    same = torch.allclose(out1[0, t], out2[0, t], atol=1e-6)
    note = "  ← 입력을 바꾼 위치라 당연" if t == 4 else ""
    print(f"위치 {t}: {'그대로 ✅' if same else '바뀜'}{note}")

Block을 3개나 통과해도 위치 0~3의 출력은 그대로입니다.
**정보가 Token 사이를 오가는 곳은 Attention뿐**이고, Attention에 마스크가 있으니 Block 전체가 미래를 보지 않아요.

## 9. LLM과의 연결

오늘 만든 `Block`은 4주차 mini GPT(커리큘럼 6-8, 6-9)에 **그대로** 들어갑니다.

```text
Token Embedding + Position Embedding     (B, T, C)
        ↓
Block × n_layer                          (B, T, C)   ← 오늘
        ↓
ln_f  (마지막 LayerNorm, Pre-LN이라서 필요)   (B, T, C)
        ↓
lm_head  (Linear C → vocab_size)         (B, T, V)
```

GPT 구조 전체는 **Day 19**에서 조립합니다.

## 10. 연습 문제

`assert`가 에러 없이 통과하면 정답입니다.

**Q1.** `Block(n_embd=48, n_head=6, block_size=32)`에 `(3, 20, 48)`을 넣으면 출력 shape는?
그리고 MLP 안에서 가장 넓어지는 순간(`mlp.fc` 직후)의 shape는? 실행하기 **전에** 예측하세요.

In [ ]:
out_shape = "?"   # TODO: 튜플로
fc_shape  = "?"   # TODO: 튜플로

blk = Block(48, 6, 32)
x = torch.randn(3, 20, 48)
assert out_shape == tuple(blk(x).shape), "Block의 출력 shape를 다시 생각해보세요"
assert fc_shape == tuple(blk.mlp.fc(blk.ln_2(x)).shape), "mlp.fc는 마지막 축을 몇 배로 늘리나요?"
print("Q1 정답!")

**Q2.** `n_embd = 64`인 Block 하나의 파라미터 개수를 **7절의 표를 보고 손으로** 계산하세요.
(Attention의 `c_attn`은 bias가 없다는 점에 주의)

In [ ]:
my_count = "?"   # TODO: 숫자 하나 (계산식으로 써도 됩니다. 예: 3 * 64 * 64 + ...)

assert my_count == count(Block(64, 4, 16)), "부품별로 나눠서 다시 계산해보세요"
print("Q2 정답!")

**Q3.** 부품은 다 만들어져 있습니다. **Pre-LN** 방식으로 `forward`를 완성하세요.

In [ ]:
class MyBlock(nn.Module):

    def __init__(self, n_embd, n_head, block_size):
        super().__init__()
        self.ln_1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, block_size)
        self.ln_2 = nn.LayerNorm(n_embd)
        self.mlp  = MLP(n_embd)

    def forward(self, x):
        # TODO: 두 줄
        return None


torch.manual_seed(1)
mine = MyBlock(32, 4, 16)
ref = Block(32, 4, 16)
ref.load_state_dict(mine.state_dict())      # 같은 Weight로 맞춤

x = torch.randn(2, 7, 32)
assert mine(x) is not None, "forward를 완성해주세요"
assert torch.allclose(mine(x), ref(x), atol=1e-6), "Pre-LN 순서를 다시 확인해보세요 (LayerNorm은 어디에?)"
print("Q3 정답!")

**Q4.** `Block`을 **3개** 쌓은 `stack`을 만들고, `x`를 통과시키세요.

In [ ]:
n_embd, n_head, block_size = 32, 4, 16

stack = None   # TODO: nn.Sequential 사용

x = torch.randn(2, 10, n_embd)
assert stack is not None, "stack을 만들어주세요"
assert len(stack) == 3 and all(isinstance(m, Block) for m in stack), "Block이 3개여야 해요"
assert tuple(stack(x).shape) == (2, 10, n_embd)
print("Q4 정답! 파라미터:", count(stack))

## 11. 체크 질문 (코드 없이 말로 답해보기)

답을 아래 셀에 적어두거나 채팅으로 알려주세요.

1. Transformer Block에는 무엇이 들어가는가? (커리큘럼 질문)
2. Attention과 FFN의 역할은 어떻게 다른가? Token 사이에 정보를 옮기는 건 어느 쪽인가?
3. Post-LN과 Pre-LN의 차이는 무엇인가? 코드 한 줄씩으로 써보세요.
4. Block의 입력 shape와 출력 shape가 같다는 게 왜 중요한가?
5. Block 하나의 파라미터 중 가장 큰 비중을 차지하는 부품은? 대략 몇 C²인가?

**내 답:**

1.
2.
3.
4.
5.